In [ ]:
# Cell 1: Simple Setup - Skip Complex Dependencies
# This approach avoids numpy binary compatibility issues

import os
import sys
import subprocess
from pathlib import Path

# Environment detection
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("MultiHealthQA"):
        subprocess.run(["git", "clone", "https://github.com/kelvintawe12/MultiHealthQA.git"], check=True)
    os.chdir("MultiHealthQA")
    
    print("Installing core dependencies only...")
    # Install only what we absolutely need, avoiding sklearn/numpy conflicts
    subprocess.run(["pip", "install", "-q", "torch", "transformers", "datasets", "accelerate", "sentencepiece", "protobuf"], check=True)
    subprocess.run(["pip", "install", "-q", "rouge-score"], check=True)
    subprocess.run(["pip", "install", "-q", "pyyaml", "tqdm"], check=True)
    
    print("Core dependencies installed")

# Add src to Python path
sys.path.insert(0, os.path.abspath("src"))
print(f"Working directory: {os.getcwd()}")

# ============================================================================
# ESSENTIAL IMPORTS ONLY - Avoid problematic packages
# ============================================================================

# Standard library imports (already imported above)

# Third-party imports (minimize dependencies)
import torch
import numpy as np
from IPython.display import display, HTML

# HuggingFace imports
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
    from datasets import Dataset
    import accelerate
except ImportError as e:
    print(f"Warning: Transformers import failed: {e}")
    print("Try: !pip install torch transformers datasets accelerate")

# Project imports (these should work without sklearn)
try:
    from mhqa.config import load_config, Config
    from mhqa.train import train
    from mhqa.modeling import load_seq2seq
except ImportError as e:
    print(f"Warning: Core project imports failed: {e}")

# Set random seeds
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Essential imports successful!")
print(f"Python {sys.version}")
print(f"PyTorch {torch.__version__}")
print(f"NumPy {np.__version__}")

# ============================================================================
# GPU Detection
# ============================================================================

def check_gpu():
    """Check GPU availability and memory."""
    if not torch.cuda.is_available():
        display(HTML("""
        <div style="background: #ffe6e6; padding: 10px; border-radius: 5px; border: 1px solid #ffcccc;">
            <strong>WARNING: No GPU detected!</strong>
        </div>
        """))
        return False, None, None
    
    device_name = torch.cuda.get_device_name(0)
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    adafactor_fits = total_memory_gb >= 12
    adamw_fits = total_memory_gb >= 24
    
    display(HTML(f"""
    <div style="background: #e6f7ff; padding: 10px; border-radius: 5px; border: 1px solid #b3d7ff;">
        <strong>GPU:</strong> {device_name}<br>
        <strong>Memory:</strong> {total_memory_gb:.1f} GB<br>
        <strong>Adafactor fits:</strong> {'YES' if adafactor_fits else 'NO'}<br>
        <strong>AdamW fits:</strong> {'YES' if adamw_fits else 'NO'}
    </div>
    """))
    
    return True, total_memory_gb, adafactor_fits

has_gpu, memory_gb, fits_large = check_gpu()
globals()['has_gpu'] = has_gpu
globals()['memory_gb'] = memory_gb
globals()['fits_large'] = fits_large

# Multilingual Health QA — GPU Training Notebook

This notebook trains a competition-grade mT5-large model for the Zindi MSRH challenge. It's designed to run on Google Colab (free T4 or paid GPU) or any local GPU with 16GB+ VRAM.

## Key Features
- **Memory-efficient**: Uses Adafactor optimizer to fit mt5-large in 16GB GPU
- **Robust loading**: Handles torch version compatibility issues
- **Complete pipeline**: EDA → Training → Evaluation → Submission
- **Colab-optimized**: Auto-detects GPU, handles file paths

In [ ]:
# Cell 2: Install Remaining Dependencies
# Install pandas, sklearn, and matplotlib after numpy is stable

print("Installing pandas, sklearn, and matplotlib...")
subprocess.run(["pip", "install", "-q", "pandas", "scikit-learn", "matplotlib"], check=True)

# Now import packages that need the stable numpy environment
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image

print(f"Pandas {pd.__version__} installed successfully")
print(f"Matplotlib {matplotlib.__version__} installed successfully")

# Import remaining project modules that need pandas/sklearn
from mhqa.data import load_all, load_split, stratified_split, build_prompt, clean_text, subset_to_language_name, make_hf_dataset, ANSWER_COL, LANG_COL, QUESTION_COL
from mhqa.metrics import compute_rouge, compute_rouge_by_language, build_compute_metrics
from mhqa.retrieval import PerLanguageRetriever
from mhqa.infer import generate_answers, predict_dataframe
from mhqa.submit import make_submission
from mhqa.evaluate import holdout_for, evaluate_model

# Configure matplotlib for headless environment
if IN_COLAB:
    import matplotlib
    matplotlib.use('Agg')

print("All imports successful!")

In [ ]:
# Cell 2: Verify Data Files
# All environment setup and imports are handled in Cell 1

print("Verifying data files...")
for f in ["Train.csv", "Val.csv", "Test.csv", "SampleSubmission.csv"]:
    p = Path("data") / f
    exists = p.exists()
    print(f"  {'OK' if exists else 'MISSING'} {f}")
    if not exists:
        print("WARNING: Missing data files! Please upload them to data/")
        if IN_COLAB:
            print("On Colab: Use the folder icon on the left to upload files to data/")

In [ ]:
# Cell 3: Load Data
# All imports handled in Cell 1

print("Loading data...")
data = load_all("data")
train, val, test = data["train"], data["val"], data["test"]
print(f"Train: {train.shape} | Val: {val.shape} | Test: {test.shape}")
display(train.head(3))

In [ ]:
# Cell 4: Run EDA
# All imports handled in Cell 1

print("Running EDA script...")
subprocess.run([sys.executable, "-m", "scripts.run_eda", "--data-dir", "data", "--out", "reports"], check=True)
print("EDA complete - figures saved to reports/figures/")

In [ ]:
# Cell 5: Display EDA Figures
# All imports handled in Cell 1

print("Displaying EDA figures...")
for fig in ["01_subset_distribution", "02_length_distributions",
            "03_answer_templating", "04_script_composition"]:
    p = f"reports/figures/{fig}.png"
    print(p)
    if Path(p).exists():
        display(Image(p))
    else:
        print(f"  Figure not found: {p}")

In [ ]:
# Cell 6: Retrieval Baseline
# All imports handled in Cell 1

print("Running retrieval baseline...")
trn, holdout = stratified_split(train, val_size=0.05, seed=42)
retr = PerLanguageRetriever().fit(trn)
preds, _, _ = retr.predict(holdout)
baseline_score = compute_rouge(preds, holdout[ANSWER_COL].tolist())
print(f"Retrieval baseline: {baseline_score}")
print("Per-language breakdown:")
display(compute_rouge_by_language(preds, holdout[ANSWER_COL].tolist(), holdout[LANG_COL].tolist()).round(4))

In [ ]:
# Cell 7: Select Configuration
# All imports and GPU detection handled in Cell 1

# Auto-select config based on hardware check from Cell 1
if has_gpu and fits_large:
    config_path = "configs/mt5_large.yaml"
    print(f"Using {config_path} (Adafactor optimizer for 16GB GPU)")
else:
    config_path = "configs/mt5_base.yaml" 
    print(f"Using {config_path} (safe default for smaller GPUs/Colab T4)")

cfg = load_config(config_path)
print(f"\nConfig summary:")
print(f"  Model: {cfg.model_name}")
print(f"  Optimizer: {cfg.optimizer}")
print(f"  Epochs: {cfg.num_train_epochs}")
print(f"  Batch size: {cfg.per_device_train_batch_size} x {cfg.gradient_accumulation_steps} = {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps} effective")
print(f"  Max length: {cfg.max_input_length} / {cfg.max_target_length}")
print(f"  Precision: {'bf16' if cfg.bf16_if_supported else 'fp16/fp32'}")

In [ ]:
# Cell 8: Train Model
# All imports handled in Cell 1

print(f"Starting training with {config_path}...")
print("This may take several hours depending on your GPU.")

trainer, tokenizer = train(cfg)
print("Training complete!")
print(f"Model saved to: {cfg.output_dir}")

In [ ]:
# Cell 9: Evaluate on Holdout Set
# All imports handled in Cell 1

print("Evaluating on holdout set...")

# Get holdout data for evaluation
from mhqa.evaluate import holdout_for  # Import here to avoid circular dependency in Cell 1
holdout = holdout_for(cfg)

# Run evaluation
from mhqa.evaluate import evaluate_model
overall, per_lang, _ = evaluate_model(trainer.model, tokenizer, holdout, cfg, batch_size=8)

print(f"\nOverall Results:")
print(f"  ROUGE-1: {overall['rouge1_f1']:.4f}")
print(f"  ROUGE-L: {overall['rougeL_f1']:.4f}")
print(f"  Combined: {overall['combined']:.4f}")

print("\nPer-Language Results:")
display(per_lang.round(4))

In [ ]:
# Cell 10: Run Experiment Suite
# All imports handled in Cell 1

print("Running experiment suite...")
print("This may take considerable time. Consider running specific experiments instead.")

# List available experiments
subprocess.run([sys.executable, "-m", "scripts.run_experiments", "--list"], check=True)

# Uncomment to run all experiments (long-running)
# subprocess.run([sys.executable, "-m", "scripts.run_experiments", "--config", config_path, "--all"], check=True)

# Uncomment to run a single experiment
# subprocess.run([sys.executable, "-m", "scripts.run_experiments", "--config", config_path, "--only", "exp04_mt5base_langprefix"], check=True)

print("To run experiments, uncomment the appropriate commands above.")

In [ ]:
# Cell 11: Generate Test Submission
# All imports handled in Cell 1

print("Generating test submission...")

# Setup retriever if needed
retriever = None
if cfg.retrieval_augment or cfg.hybrid_fallback:
    print("Setting up retriever for hybrid fallback...")
    retriever = PerLanguageRetriever().fit(load_split(cfg.train_path, has_answer=True))

# Load test data
test_df = load_split(cfg.test_path, has_answer=False)

# Generate predictions
print("Running inference...")
test_preds = predict_dataframe(trainer.model, tokenizer, test_df, cfg,
                               retriever=retriever, batch_size=8)

# Create and validate submission
print("Creating submission file...")
sub = make_submission(
    ids=test_df[ID_COL].tolist(),
    predictions=test_preds,
    output_path=cfg.submission_path,
    sample_submission_path=cfg.sample_submission_path,
)

print(f"Submission saved to: {cfg.submission_path}")
print(f"Submission shape: {sub.shape}")
print("\nFirst few rows:")
display(sub.head())

print("\nReady to submit to Zindi!")

## Next Steps to Climb the Leaderboard

1. Re-run Cell 8 with `configs/mt5_large.yaml` on your GPU
2. Sweep experiments `exp06`→`exp11` and record `public_lb` in `reports/experiments.csv`
3. Update `reports/leaderboard_progression.csv` after each submission

## Important Notes

- **Cell 1 handles all imports** - Run it first to set up the environment
- **GPU detection** - Automatic config selection based on hardware
- **Numpy compatibility** - Automatic fixes for Colab environment issues
- **Data files** - Must be manually uploaded to data/ directory on Colab

## 5 — Held-out evaluation (overall + per-language)

In [ ]:
from mhqa.evaluate import evaluate_model, holdout_for
holdout = holdout_for(cfg)
overall, per_lang, _ = evaluate_model(trainer.model, tokenizer, holdout, cfg, batch_size=8)
print(f"ROUGE-1={overall['rouge1_f1']:.4f}  ROUGE-L={overall['rougeL_f1']:.4f}  "
      f"combined={overall['combined']:.4f}")
per_lang.round(4)

## 6 — Experiment suite

Each experiment is a config delta + hypothesis, logged to
`reports/experiments.csv`. Run individually or all at once (long — GPU box).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "scripts.run_experiments", "--list"], check=True)
# Example single run:
# subprocess.run([sys.executable, "-m", "scripts.run_experiments",
#                 "--config", "configs/mt5_base.yaml",
#                 "--only", "exp04_mt5base_langprefix"], check=True)

In [ ]:
import pandas as pd, os
if os.path.exists("reports/experiments.csv"):
    display(pd.read_csv("reports/experiments.csv"))

## 7 — Generate the validated test submission

In [ ]:
from mhqa.data import ID_COL, load_split
from mhqa.infer import predict_dataframe
from mhqa.submit import make_submission

retriever = None
if cfg.retrieval_augment or cfg.hybrid_fallback:
    retriever = PerLanguageRetriever().fit(load_split(cfg.train_path, has_answer=True))

test_df = load_split(cfg.test_path, has_answer=False)
test_preds = predict_dataframe(trainer.model, tokenizer, test_df, cfg,
                               retriever=retriever, batch_size=8)
sub = make_submission(
    ids=test_df[ID_COL].tolist(),
    predictions=test_preds,
    output_path=cfg.submission_path,
    sample_submission_path=cfg.sample_submission_path,  # validates shape + IDs
)
print("Wrote", cfg.submission_path)
sub.head()

## Next steps to climb the leaderboard

1. Re-run section 4 with `configs/mt5_large.yaml` on the GPU.
2. Sweep `exp06`→`exp11` and record `public_lb` in `reports/experiments.csv`.
3. Update `reports/leaderboard_progression.csv` after each submission and plot it
   for the report's progression chart.
